In [1]:
from torch import nn, optim
from torch.utils.data.dataloader import DataLoader
from transformers import AutoTokenizer
import torch 
import math
import time
from contextlib import nullcontext
import warnings
import os
import sys
sys.path.append(os.path.join(os.getcwd(), ".."))
from model.LMConfig import LMConfig
from model.model import MiniMindLM
from model.dataset import SFTDataset

warnings.filterwarnings('ignore')

/Users/kevinlights/app/miniforge3/envs/minimind/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
learning_rate = 5e-4
ctx = nullcontext()
accumulation_steps = 8
device = "mps"
grad_clip = 1.0
log_interval = 100
save_interval = 50
total_epochs = 1
save_dir = "output"

os.makedirs(save_dir, exist_ok=True)

def train(epoch):
    start_time = time.time()
    loss_func = nn.CrossEntropyLoss(reduction='none')
    for step, (X, Y, loss_mask) in enumerate(train_loader):
        X = X.to(device)
        Y = Y.to(device)
        loss_mask = loss_mask.to(device)
        # X = X.to(device, dtype=torch.float32)
        # Y = Y.to(device, dtype=torch.float32)
        # loss_mask = loss_mask.to(device, dtype=torch.float32)

        current_step = epoch * iter_per_epoch + step
        total_steps = total_epochs * iter_per_epoch
        
        # 余弦退火学习率动态调整，最低学习率 lr/10，右边公式让学习率按照余弦曲线从初始值下降到最低值
        # current_step/total_steps 从 0 逐渐接近 1，余弦值从 1 降到 -1，整个表达式就从 lr 慢慢降到 lr/10
        # 动态调整学习率的好处是：
        # 开始阶段学习率下降较慢，有利于模型在初期快速学习 
        # 后期学习率缓慢接近最小值，有利于模型精细调整
        # 避免了学习率突变对训练造成的影响
        # 相比单纯线性下载更符合训练过程的实际需求
        lr = learning_rate / 10 + 0.5 * learning_rate * (1 + math.cos(math.pi * current_step / total_steps))

        # 有些模型不同部分可能使用不同的学习率，所有会有多个参数组
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        with ctx:
            # res 通常包含 logits: 模型的原始预测值，尚未经过 softmax 归一化
            # aux_loss: 辅助损失，某些模型会有额外的正则化损失
            res = model(X)
            loss = loss_func(
                res.logits.view(-1, res.logits.size(-1)), # 调整 logits 形状，把 logits 从 [batch_size, seq_len, vocab_size] 变成 [batch_size * seq_len, vocab_size]，这是因为交叉熵损失通常要求 logits 是二维的
                Y.view(-1) # 调整标签形状，从 [batch_size, seq_len] 变成 [batch_size * seq_len]
            ).view(Y.size()) # 恢复 loss 形状，恢复成 [batch_size, seq_len]
            # 对损失值进行加权平均，只计算有效部分，忽略 padding 等无效位置
            loss = (loss * loss_mask).sum() / loss_mask.sum()
            # 加上模型的辅助扣件
            loss += res.aux_loss
            # 按梯度累积步数缩放损失值，相当于把多个小 batch 的损失平均分摊到每一步，这样在 loss.backward() 时，梯度会被自动累加，但当前损失值看起来更小，避免数值过大
            loss = loss / accumulation_steps
        
        loss.backward()

        if (step + 1) % accumulation_steps == 0:
            # 手动梯度裁剪，防止梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

            # for name, param in model.named_parameters():
            #     if param.grad is not None:
            #         print(f"{name}: max_grad={param.grad.abs().max()}, mean_grad={param.grad.mean()}")
            
            if torch.isnan(loss).any() or torch.isinf(loss).any():
                print("Loss contains NaN/Inf!")
            else:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        if step % log_interval == 0:
            spend_time = time.time() - start_time
            print(f"epoch: {epoch+1}/{total_epochs} step: {step} epoch_steps: {iter_per_epoch} loss: {loss.item() * accumulation_steps:.3f} lr: {optimizer.param_groups[-1]['lr']} spend_time: {spend_time/60:.1f}m left_time: {iter_per_epoch/(step+1)*spend_time/60-spend_time/60:.1f}m")

        # dist.get_rank() 是获取当前进程在分布式训练环境中的唯一标识，从 0 开始，0 是主进程，负责协调，保存模型等
        # if (step + 1) % save_interval == 0 and dist.get_rank() == 0:
        if (step + 1) % save_interval == 0:
            model.eval()
            ckp = f"{save_dir}/full_sft_{lm_config.dim}.pth"
            state_dict = model.state_dict()
            torch.save(state_dict, ckp)
            model.train()

lm_config = LMConfig(
    dim=512,
    n_layers=8,
    n_heads=8,
    n_kv_heads=2,
    vocab_size=6400,
    hidden_dim=None,
    multiple_of=64,
    norm_eps=0.00001,
    max_seq_len=512,
    rope_theta=1000000,
    dropout=0,
    flash_attn=True,
    use_moe=False,
    num_experts_per_tok=2,
    n_routed_experts=4,
    n_shared_experts=True,
    scoring_func="softmax",
    aux_loss_alpha=0.1,
    seq_aux=True,
    norm_topk_prob=True,
)
tokenizer = AutoTokenizer.from_pretrained("../model/minimind_tokenizer")
model = MiniMindLM(lm_config)
# ckp = f"output/pretrain_64.pth"
ckp = f"../downloads/pretrain_512.pth"
state_dict = torch.load(ckp, map_location=device)
model.load_state_dict(state_dict, strict=False)
print(f'LLM总参数量：{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.3f} 百万')
model = model.to(device)

torch.manual_seed(1337)

train_ds = SFTDataset("../dataset/sft_mini_512_mini.jsonl", tokenizer, max_length=lm_config.max_seq_len)
train_loader = DataLoader(
    train_ds,
    batch_size=8,
    pin_memory=True,
    drop_last=False,
    shuffle=False,
    num_workers=1,
    sampler=None
)

optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

iter_per_epoch = len(train_loader)

for epoch in range(total_epochs):
    train(epoch)

LLM总参数量：25.830 百万
epoch: 1/1 step: 0 epoch_steps: 1250 loss: 4.507 lr: 0.00055 spend_time: 0.0m left_time: 36.4m
epoch: 1/1 step: 100 epoch_steps: 1250 loss: 7.000 lr: 0.0005421457902821578 spend_time: 0.9m left_time: 9.8m
epoch: 1/1 step: 200 epoch_steps: 1250 loss: 6.054 lr: 0.0005190766700109659 spend_time: 1.7m left_time: 8.9m
epoch: 1/1 step: 300 epoch_steps: 1250 loss: 5.530 lr: 0.0004822421568553529 spend_time: 2.5m left_time: 8.0m
epoch: 1/1 step: 400 epoch_steps: 1250 loss: 5.301 lr: 0.00043395669874474917 spend_time: 3.4m left_time: 7.1m
epoch: 1/1 step: 500 epoch_steps: 1250 loss: 5.100 lr: 0.0003772542485937369 spend_time: 4.2m left_time: 6.3m
epoch: 1/1 step: 600 epoch_steps: 1250 loss: 4.428 lr: 0.0003156976298823284 spend_time: 5.0m left_time: 5.5m
epoch: 1/1 step: 700 epoch_steps: 1250 loss: 4.449 lr: 0.00025315467135356894 spend_time: 5.9m left_time: 4.6m
epoch: 1/1 step: 800 epoch_steps: 1250 loss: 4.113 lr: 0.00019355517710873183 spend_time: 6.7m left_time: 3.8m
epoc

In [14]:
tokenizer = AutoTokenizer.from_pretrained("../model/minimind_tokenizer")
# ckp = f"{save_dir}/full_sft_{lm_config.dim}.pth"
# ckp = f"../downloads/pretrain_512.pth"
ckp = f"../downloads/full_sft_512_zero.pth"
model = MiniMindLM(
    LMConfig(
        dim=lm_config.dim,
        n_layers=lm_config.n_layers,
        max_seq_len=lm_config.max_seq_len,
        use_moe=lm_config.use_moe,
    )
)
state_dict = torch.load(ckp, map_location=device)
model.load_state_dict({k: v for k, v in state_dict.items() if 'mask' not in k}, strict=True)
model = model.eval().to(device)

# messages = [{"role": "user", "content": "秦始皇"}]
# messages = [{"role": "user", "content": "详细的介绍光速的物理概念。"}]
messages = [{"role": "user", "content": "生成一个描述夏天的短小故事。"}]
# messages = [{"role": "user", "content": "牛奶是一种透明的液体吗？"}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
with torch.no_grad():
    x = torch.tensor(tokenizer(prompt)["input_ids"], device=device).unsqueeze(0)
    outputs = model.generate(
        x, 
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=512,
        temperature=0.85,
        top_p=0.85,
        stream=True,
        pad_token_id=tokenizer.pad_token_id,
    )
    print('🤖️: ', end='')
    try:
        history_idx = 0
        for y in outputs:
            answer = tokenizer.decode(y[0].tolist(), skip_special_tokens=True)
            if (answer and answer[-1] == '�') or not answer:
                continue
            print(answer[history_idx:], end='', flush=True)
            history_idx = len(answer)
    except StopIteration:
        print("No answer")
    print('\n')
messages.append({"role": "assistant", "content": answer})
print(messages)

🤖️: 在炎热的夏天，小鸟们都在欢快地歌唱。小鸟们开始飞快地奔跑着，发出欢快的声音。远处，传来阵阵阵鸟鸣声，如同一首美妙的诗。人们穿着轻便的衣服，在沙滩上玩耍，享受着阳光和海风。傍晚时分，天空渐渐变暗，晚霞渐渐染红了整个天空，这就是夏天的美好。

[{'role': 'user', 'content': '生成一个描述夏天的短小故事。'}, {'role': 'assistant', 'content': '在炎热的夏天，小鸟们都在欢快地歌唱。小鸟们开始飞快地奔跑着，发出欢快的声音。远处，传来阵阵阵鸟鸣声，如同一首美妙的诗。人们穿着轻便的衣服，在沙滩上玩耍，享受着阳光和海风。傍晚时分，天空渐渐变暗，晚霞渐渐染红了整个天空，这就是夏天的美好。'}]
